# Wav2Vec 2.0

In [2]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from transformers import AutoConfig

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

# =========================
# 1. Setup Data
# =========================

DATA_PATH = "./data/CREMA-D/"
EMOTIONS = {'ANG': 0, 'DIS': 1, 'FEA': 2, 'HAP': 3, 'NEU': 4, 'SAD': 5}

data = []
for filename in os.listdir(DATA_PATH):
    if filename.endswith(".wav"):
        parts = filename.split('_')
        if parts[2] in EMOTIONS:
            data.append({
                "path": os.path.join(DATA_PATH, filename),
                "speaker": parts[0],
                "label": EMOTIONS[parts[2]]
            })

df = pd.DataFrame(data)

# =========================
# 2. Speaker split
# =========================

unique_speakers = df['speaker'].unique().tolist()
train_speakers, test_speakers = train_test_split(
    unique_speakers, test_size=0.2, random_state=42
)
train_df = df[df['speaker'].isin(train_speakers)].reset_index(drop=True)
test_df  = df[df['speaker'].isin(test_speakers)].reset_index(drop=True)

print(f"Total: {len(df)} | Train: {len(train_df)} | Test: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)

# =========================
# 3. Feature extractor
# =========================

model_id = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

MAX_DURATION = 3.0
MAX_LENGTH   = int(MAX_DURATION * feature_extractor.sampling_rate)  # 48 000 samples

# =========================
# 4. Data augmentation helpers
# =========================
# Only applied during training — keeps the test set clean.
# Each function operates on a 1-D float32 numpy array at 16 kHz.

def add_gaussian_noise(array, snr_db=None):
    """
    Additive white Gaussian noise at a random SNR between 15–35 dB.
    Lower SNR = more noise; 15 dB is clearly audible, 35 dB is almost
    imperceptible — this range keeps the emotion signal intact.
    """
    if snr_db is None:
        snr_db = random.uniform(15, 35)
    signal_power = np.mean(array ** 2) + 1e-9
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise        = np.random.normal(0, np.sqrt(noise_power), array.shape)
    return (array + noise).astype(np.float32)


def time_mask(array, sr=16000, max_ms=200):
    """
    SpecAugment-style time masking: zero out a random contiguous
    segment up to `max_ms` milliseconds long. Teaches the model
    to classify from partial utterances — useful because CREMA-D
    sentences vary in length.
    """
    max_samples = int(sr * max_ms / 1000)
    mask_len    = random.randint(0, max_samples)
    if mask_len == 0 or len(array) <= mask_len:
        return array
    start = random.randint(0, len(array) - mask_len)
    out = array.copy()
    out[start:start + mask_len] = 0.0
    return out


def pitch_shift(array, sr=16000, max_semitones=2):
    """
    Random pitch shift ±2 semitones. Semitones are a log scale, so
    ±2 changes pitch by ~12 % without altering tempo — small enough
    that the emotional prosody is preserved.
    """
    n_steps = random.uniform(-max_semitones, max_semitones)
    return librosa.effects.pitch_shift(array, sr=sr, n_steps=n_steps).astype(np.float32)


def augment(array, sr=16000):
    """
    Apply each augmentation independently with 50 % probability so
    the model sees many combinations. The order (noise → mask → pitch)
    matters slightly: noise first makes masking more realistic.
    """
    if random.random() < 0.5:
        array = add_gaussian_noise(array)
    if random.random() < 0.5:
        array = time_mask(array, sr)
    if random.random() < 0.5:
        array = pitch_shift(array, sr)
    return array

# =========================
# 5. Preprocessing
# =========================

def make_preprocess(is_train):
    """
    Returns a batched preprocessing function.
    `is_train=True` enables augmentation; test/eval skips it.
    """
    def preprocess_function(examples):
        audio_arrays = []
        for path in examples["path"]:
            array, _ = librosa.load(path, sr=16000)
            if is_train:
                array = augment(array)
            audio_arrays.append(array)

        inputs = feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
        )
        inputs["labels"] = examples["label"]
        return inputs
    return preprocess_function


encoded_train = train_dataset.map(
    make_preprocess(is_train=True),
    remove_columns=["path", "speaker"],
    batched=True,
    batch_size=4,
)
encoded_test = test_dataset.map(
    make_preprocess(is_train=False),
    remove_columns=["path", "speaker"],
    batched=True,
    batch_size=4,
)

# =========================
# 6. Label smoothing loss
# =========================
# CrossEntropyLoss with label_smoothing=0.1 softens the hard one-hot
# targets to (1-ε) for the true class and ε/(K-1) for others.
# This penalises overconfident predictions and regularises the head.

class SmoothedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# =========================
# 7. Metrics
# =========================

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=eval_pred.label_ids)

# =========================
# 8. Model + dropout tuning
# =========================
# wav2vec2-base exposes three dropout knobs:
#   hidden_dropout_prob     — dropout inside the transformer layers
#   feat_proj_dropout       — dropout on the CNN feature projection
#   classifier_proj_size /  — the classification head itself
# We also add a custom head dropout via `final_dropout`.
# These values are mild so the pre-trained representations aren't
# over-regularised — wav2vec2 is already well-trained.

num_labels = len(EMOTIONS)

# Build config first, then patch dropout values, then load model from config
config = AutoConfig.from_pretrained(
    model_id,
    num_labels=num_labels,
    id2label={str(v): k for k, v in EMOTIONS.items()},
    label2id={k: v for k, v in EMOTIONS.items()},
)

# Wav2Vec2-specific dropout parameter names
config.hidden_dropout      = 0.1   # dropout inside transformer layers
config.attention_dropout   = 0.1   # dropout on attention weights
config.feat_proj_dropout   = 0.1   # CNN feature projection dropout
config.final_dropout       = 0.2   # classification head input dropout
config.layerdrop           = 0.05  # stochastic layer drop (additional regularisation)

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    config=config,
    ignore_mismatched_sizes=True,
)
# =========================
# 9. Training config
# =========================

training_args = TrainingArguments(
    output_dir="./wav2vec2-crema-d-emotion",

    # Evaluation & checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # Optimiser
    learning_rate=3e-5,
    # Effective batch = 8 × 4 = 32. Accumulation reduces memory pressure
    # while maintaining a larger effective batch for stable gradients.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    # Schedule
    num_train_epochs=15,          # more epochs — early stopping will cut this short
    warmup_ratio=0.1,

    # Regularisation
    weight_decay=0.01,            # L2 on all non-bias, non-LayerNorm parameters
    max_grad_norm=1.0,            # gradient clipping — prevents exploding gradients

    # Logging
    logging_steps=20,
    report_to="none",             # set to "wandb" if you use W&B
)

# =========================
# 10. Trainer + early stopping
# =========================
# Patience=3 means training stops if eval accuracy doesn't improve
# for 3 consecutive epochs — prevents overfitting on small datasets.

trainer = SmoothedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# 11. Train
# =========================

trainer.train()

Total: 7442 | Train: 5890 | Test: 1552


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 84053.39it/s]
Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
projector.bias               | MISSING    | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprec

Epoch,Training Loss,Validation Loss,Accuracy
1,6.347600,1.557453,0.392397
2,5.692951,1.370515,0.511598
3,5.045112,1.191890,0.621134
4,4.067983,1.158692,0.644974
5,4.187967,1.192216,0.652706
6,3.801548,1.192158,0.662371
7,3.664297,1.125360,0.691366
8,3.072875,1.203048,0.681701
9,2.952319,1.174313,0.685567
10,2.892025,1.193574,0.693943


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.53it/s]


TrainOutput(global_step=2775, training_loss=3.6855283981185774, metrics={'train_runtime': 2170.1896, 'train_samples_per_second': 40.711, 'train_steps_per_second': 1.279, 'total_flos': 2.4063177384864e+18, 'train_loss': 3.6855283981185774, 'epoch': 15.0})

In [3]:
# =========================
# 12. Evaluate
# =========================

predictions   = trainer.predict(encoded_test)
preds         = np.argmax(predictions.predictions, axis=1)
labels        = predictions.label_ids
test_accuracy = (preds == labels).mean()
print("✅ Test Accuracy:", test_accuracy)
print("Full metrics:", predictions.metrics)

✅ Test Accuracy: 0.6997422680412371
Full metrics: {'test_loss': 1.2578434944152832, 'test_accuracy': 0.6997422680412371, 'test_runtime': 21.2138, 'test_samples_per_second': 73.16, 'test_steps_per_second': 9.145}
